# MIDI（累加式）+ 弦号 → 尤克里里把位

**`data.txt` 和弦片段格式**：`[p0^d1^d2^...]|[s0^s1^s2^...]`

- **音高**：第一个数为 MIDI 基准，其后每一项为**相对前一累计结果的增量**（逐步累加）。  
  例：`[60^4^3^2]` → `[60, 64, 67, 69]`。
- **弦号**：自下而上为 `0,1,2,3`；与音高一一对应。  
  例：`[2^1^3^0]` 表示第 1 个音在弦 2，第 2 个在弦 1，…
- **标准调弦 GCEA**（对应空弦 MIDI）：弦 0=A4(69)、弦 1=E4(64)、弦 2=C4(60)、弦 3=G4(67)。
- **固定 4 维输出**：始终按弦 `0,1,2,3` 输出 `[f0,f1,f2,f3]`，未出现的弦补 `0`。  
  例：只按 0 弦 3 品 → `[3,0,0,0]`。

例：`[60^4^3^2]|[2^1^3^0]` → 四根弦均为空弦 → **`[0,0,0,0]`**。

In [5]:
from __future__ import annotations

import re
from pathlib import Path

# 优先：内核工作目录下的 data.txt；否则尝试仓库 data/yousician/data.txt
_cwd = Path.cwd()
DATA_TXT = _cwd / "data.txt"
if not DATA_TXT.is_file():
    DATA_TXT = _cwd / "data" / "yousician" / "data.txt"

In [6]:
# 弦索引 0–3：自下而上；标准尤克里里 reentrant GCEA 空弦 MIDI
OPEN_MIDI_BY_STRING = {0: 69, 1: 64, 2: 60, 3: 67}

_CHORD_RE = re.compile(
    r"\[([0-9]+(?:\^[0-9]+)*)\]\|\[([0-9]+(?:\^[0-9]+)*)\]"
)


def parse_cumulative_pitches(inner: str) -> list[int]:
    parts = [int(x) for x in inner.split("^")]
    if not parts:
        return []
    out = [parts[0]]
    for d in parts[1:]:
        out.append(out[-1] + d)
    return out


def parse_strings(inner: str) -> list[int]:
    return [int(x) for x in inner.split("^")]


def chord_to_fret_token(pitch_bracket: str, string_bracket: str) -> str:
    """输入如 '60^4^3^2' / '2^1^3^0'，输出固定 4 维把位 '[f0,f1,f2,f3]'。"""
    mids = parse_cumulative_pitches(pitch_bracket)
    strs = parse_strings(string_bracket)
    if len(mids) != len(strs):
        raise ValueError(f"音高数 {len(mids)} 与弦数 {len(strs)} 不一致: {mids!r} vs {strs!r}")

    # 固定按弦 0,1,2,3 输出；未出现弦保持 0（空弦）
    fret_by_string = [0, 0, 0, 0]
    for m, s in zip(mids, strs):
        if s not in OPEN_MIDI_BY_STRING:
            raise ValueError(f"非法弦号 {s}（应为 0–3）")
        fret = m - OPEN_MIDI_BY_STRING[s]
        fret_by_string[s] = fret

    return "[" + ",".join(str(f) for f in fret_by_string) + "]"


def convert_chord_match(m: re.Match) -> str:
    p, s = m.group(1), m.group(2)
    return chord_to_fret_token(p, s)


def line_chords_to_frets(line: str) -> str:
    """将一行中所有 [..]|[..] 和弦模式替换为固定 4 维把位 [f0,f1,f2,f3]。"""
    return _CHORD_RE.sub(lambda m: convert_chord_match(m), line)

In [7]:
# 用户给定示例：四根弦都空弦
assert chord_to_fret_token("60^4^3^2", "2^1^3^0") == "[0,0,0,0]"

# 只按 0 弦 3 品，其余空弦
assert chord_to_fret_token("72", "0") == "[3,0,0,0]"

# 另一常见和弦：0 弦高 3 品，其余空弦
print("[60^4^3^5]|[2^1^3^0] ->", chord_to_fret_token("60^4^3^5", "2^1^3^0"))  # [3,0,0,0]

# 整段替换示例
sample = "inf|[60^4^3^2]|[2^1^3^0],70|[69]|[0]"
print(line_chords_to_frets(sample))

[60^4^3^5]|[2^1^3^0] -> [3,0,0,0]
inf|[0,0,0,0],70|[0,0,0,0]


In [8]:
def preview_data_txt(path: Path, max_lines: int = 3, max_chars: int = 400) -> None:
    if not path.is_file():
        print(f"未找到文件: {path}")
        return
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            conv = line_chords_to_frets(line.rstrip("\n"))
            snippet = conv[:max_chars] + ("..." if len(conv) > max_chars else "")
            print(f"--- line {i + 1} (前 {max_chars} 字符) ---")
            print(snippet)


preview_data_txt(DATA_TXT)

--- line 1 (前 400 字符) ---
uSJBJX,38608
--- line 2 (前 400 字符) ---
inf|[0,0,0,0],inf|[0,1,0,0],70|[0,0,0,0],70|[0,0,0,0],70|[0,1,0,0],70|[0,0,2,0],70|[0,0,2,0],70|[0,1,0,0],70|[0,0,0,0],70|[0,0,0,0],70|[0,1,0,0],70|[0,0,2,0],70|[0,0,2,0],70|[0,0,0,0],70|[3,0,0,0],70|[0,3,0,0],70|[0,1,0,0],70|[0,0,2,0],70|[0,0,2,0],70|[0,0,0,0],70|[3,0,0,0],70|[0,3,0,0],70|[0,1,0,0],70|[0,0,2,0],70|[0,0,2,0],inf|[0,1,0,0],70|[0,0,0,0],70|[0,0,0,0],70|[0,3,0,0],70|[0,1,0,0],70|[0,0...
--- line 3 (前 400 字符) ---
[60^4^3^2],[65],[69],[64],[65],[62],[62],[65],[69],[64],[65],[62],[62],[69],[72],[67],[65],[62],[62],[69],[72],[67],[65],[62],[62],[65],[69],[64],[67],[65],[62],[62],[65],[69],[64],[67],[65],[62],[62],[69],[72],[67],[67],[65],[62],[62],[69],[72],[67],[67],[65],[62],[62],[70],[67],[64],[67],[69],[65],[62],[70],[67],[64],[67],[65],[62],[62],[70],[67],[64],[67],[69],[65],[62],[70],[67],[64],[67],[65]...


**说明**：当前正则同时支持单音和和弦：

- `70|[69]|[0]` 会被转成 `70|[0,0,0,0]`
- `[60^4^3^2]|[2^1^3^0]` 会被转成 `[0,0,0,0]`

输出始终是固定 4 维（按弦 0,1,2,3）。